# 6. Paradigm Shift

## 6.1 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   Bajar el competencia_01_crudo al Google Drive y tambien al disco local de la virtual machine que está corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf" /content/buckets/b1


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/dmeyf2026-9c6f/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "competencia_01_crudo.csv"




---



##  6.2 Generacion de la clase_ternaria

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

In [ ]:
require( "data.table" )

# leo el dataset
dataset <- fread("/content/datasets/competencia_01_crudo.csv" )

# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]


# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

fwrite( dataset,
    file =  "/content/datasets/competencia_01.csv.gz",
    sep = ","
)

In [ ]:
setorder( dataset, foto_mes, clase_ternaria, numero_de_cliente)
dataset[, .N, list(foto_mes, clase_ternaria)]



---



## 6.3 fenomeno que NO deberia suceder

"El arbol de decisión que encuentra una Bayesian Optimizacion es el optimo, **no** overfitea".
<br>Certain ideas have been accepted as true without sufficient carfeful thought


Se agregarán *variables canarito* al dataset, se entrenará el arbol con los mejores hiperparámetros encontrados, y se analizará si los canaritos aparecen en algun split.

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

### 6.3.1  carga manual de hiperparámetros
Aqui debe cargar SU semilla primigenia y
<br> SUS mejores hiperparámetros que encontró para el ARBOL DE DECISION, ya sea por Grid Search o  Bayesian Optimization

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191

PARAM$rpart$cp <- -1
PARAM$rpart$maxdepth <- 6
PARAM$rpart$minsplit <- 50
PARAM$rpart$minbucket <- 5

### 6.3.2  corrida

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp6300"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/competencia_01.csv.gz")

In [ ]:
# datos entrenamiento
dtrain <- dataset[ foto_mes==202104,]

In [ ]:
# uso esta semilla para los canaritos
set.seed(PARAM$semila_primigenia)

# agrego los siguientes canaritos
for( i in 1:154 ) dtrain[ , paste0("canarito", i ) :=  runif( nrow(dtrain)) ]

Espere a que el profesor le indique cuando correr esta celda, se hará en forma SINCRONIZADA en el aula
<br> aprox 4 min

In [ ]:
# Entreno el modelo

modelo <- rpart(formula= "clase_ternaria ~ .",
  data= dtrain,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart
)


In [ ]:
# genero un pdf con el dibujo del arbol

pdf(file = "arbol_canaritos.pdf", width=28, height=4)
prp(modelo, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

vaya a su Google Drive
<br> busque la carpeta **My Drive / labo1 / exp / exp6300**
<br> baje el archivo **arbol_canaritos.pdf**  a su laptop
<br> abra el .pdf con el Acrobat Reader
<br> y dentro del .pdf busque splits hechos en alguna de las nuevas variables canaritos



---



## 6.4  rpart tradicional

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191

In [ ]:
# Hiperparametros optimos encontrados en una Bayesian Optimization
PARAM$peso <- 15.9742385635332
PARAM$rpart$cp <- -1
PARAM$rpart$maxdepth <- 27
PARAM$rpart$minsplit <- 1684
PARAM$rpart$minbucket <- 447

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp6400"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

# lectura del dataset
dataset <- fread("/content/datasets/competencia_01.csv.gz")

In [ ]:
# elimino los campos que causan Data Drifting
dataset[, cprestamos_personales := NULL ]
dataset[, mprestamos_personales := NULL ]

In [ ]:
# Final Train
dfinal_train <- dataset[ foto_mes==202104,]

In [ ]:
# clase binaria
dfinal_train[, clase_binaria2 := ifelse( clase_ternaria=="CONTINUA", "NEG", "POS" ) ]
dfinal_train[, clase_ternaria := NULL ]

In [ ]:
pesos <- dfinal_train[, ifelse( clase_binaria2=="POS", PARAM$peso, 1.0 ) ]

corre en 2 minutos

In [ ]:
modelo_final <- rpart(formula= "clase_binaria2 ~ .",
  data= dfinal_train,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart,
  weights= pesos
)


In [ ]:
#  future
dfuture <- dataset[ foto_mes==202106,]

In [ ]:
# aplico el modelo a los datos del futuro
prediccion <- predict(modelo_final,
  dfuture,
  type= "prob"
)


In [ ]:
dfuture[, prob := prediccion[,"POS"]]
dfuture[, gan := ifelse(clase_ternaria == "BAJA+2", 1.0725, -0.0275)]

In [ ]:
setorder( dfuture, -prob)
dfuture[, gan_acum:= cumsum(gan)]
dfuture[, gan_suave := frollmean(gan_acum, n=501, align="center", na.rm=TRUE)]

In [ ]:
# Mejor Ganancia
dfuture[, max(gan_suave, na.rm=TRUE)]

La máxima ganancia promedio fue de  360.8 millones



---



## 6.6 Paradigm Shift

Pasamos a trabajar con una clase  Binaria


*   POS = { BAJA+1, BAJA+2 }
*   NEG = { CONTINUA }




ahora la probabilidad que devuelve el modelo es de POS,
<br> ya no es la de BAJA+2,
<br> ya no puedo cortar por ella
<br> debo cortar por cantidad de envios !

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

Aqui debe cargar SU semilla primigenia

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191

PARAM$peso <- 500

# Dejo crecer el arbol sin ninguna limitacion
# sin limite de altura ( 30 es el maximo que permite rpart )
# sin limite de minsplit ( 2 es el minimo natural )
# sin limite de minbukcet( 1 es el minimo natural )
# ya aprendimos que cp debe ser negativo
PARAM$rpart$cp <- -1
PARAM$rpart$maxdepth <- 16 # deberia ser 31, por velocidad en clase se baja  16
PARAM$rpart$minsplit <- 2
PARAM$rpart$minbucket <- 1

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp6600"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/competencia_01.csv.gz")

In [ ]:
# elimino los campos que causan Data Drifting
dataset[, cprestamos_personales := NULL ]
dataset[, mprestamos_personales := NULL ]

In [ ]:
# uso esta semilla para los canaritos
set.seed(PARAM$semila_primigenia)

for( i in 1:155 ) {
  dataset[, paste0("canarito", i ) :=  runif( nrow(dataset)) ]
}

In [ ]:
# datos de training
dtrain <- dataset[foto_mes == 202104]

In [ ]:
# clase binaria
dtrain[, clase_binaria2 := ifelse( clase_ternaria=="CONTINUA", "NEG", "POS" ) ]
dtrain[, clase_ternaria := NULL ]

la siguiente celda corre en 6 minutos

In [ ]:
# Entreno el modelo
pesos <- dtrain[, ifelse( clase_binaria2=="POS", PARAM$peso, 1.0 ) ]

modelo_original <- rpart(formula= "clase_binaria2 ~ .",
  data= dtrain,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart,
  weights= pesos
)


In [ ]:
# hago el pruning de los canaritos
# haciendo un hackeo a la estructura  modelo_original$frame
# -666 es un valor arbritrariamente negativo que jamas es generado por rpart

modelo_original$frame[
  modelo_original$frame$var %like% "canarito",
  "complexity"
] <- -666

modelo_pruned <- prune(modelo_original, -666)

In [ ]:
# genero un pdf con el dibujo del arbol

pdf(file= "stopping_at_canaritos.pdf", width=28, height=4)
prp(modelo_pruned, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

In [ ]:
# datos del futuro
dfuturePS <- dataset[foto_mes == 202106]

In [ ]:
# scoring, aplico el modelo a los datos del futuro
prediccion <- predict(modelo_pruned,
  dfuturePS,
  type= "prob"
)

In [ ]:
dfuturePS[, prob := prediccion[,"POS"]]
dfuturePS[, gan := ifelse(clase_ternaria == "BAJA+2", 1.0725, -0.0275)]

In [ ]:
setorder( dfuturePS, -prob)
dfuturePS[, gan_acum:= cumsum(gan)]
dfuturePS[, gan_suave := frollmean(gan_acum, n=501, align="center", na.rm=TRUE)]

In [ ]:
# Mejor Ganancia
dfuturePS[, max(gan_suave, na.rm=TRUE)]

vaya a su Google Drive
<br> busque la carpeta **My Drive /  labo1 / exp / exp6600**
<br> baje el archivo **stopping_at_canaritos.pdf**  a su laptop
<br> abra el .pdf con el Acrobat Reader




---

